# BATDiff alone on a natural photograph (sanity check)

Supervisor request, 9 September 2026: before going back to OCT, show that BATDiff
can reconstruct a **natural image**, not noise.

This notebook is **BATDiff alone**. It does not use a DIP reference.

| Setting | Value |
|---|---|
| Image | DIV2K `0801.png` **full** photograph (no crop) |
| How the small image was made | Gaussian blur, then `::4` (every fourth row/column) |
| `sr_factor` | **4** |
| `x_ref` | bicubic *enlargement* of `lr.png` (published BATDiff behaviour) |

## Before you start

1. On your Mac, run Part A in `notes/how_to_natural_sanity.md` so you have
   `outputs/sanity/div2k_filtered_stride_x4/lr.png` and `hr.png`.
2. **Runtime → Change runtime type → T4 GPU**.
3. Run the cells **from top to bottom**. Do not skip.

Leave `SMOKE_TEST = True` for the first pass (a few minutes; the picture may look
like noise). If that works, set it to `False`, restart the runtime, and run again.

In [ ]:
#@title Step 0 — check the runtime has a GPU
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU visible.\n"
        "Fix: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, "
        "then run this cell again."
    )

total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU:   {torch.cuda.get_device_name(0)}")
print(f"VRAM:  {total_gb:.1f} GB")
print(f"torch: {torch.__version__}")

## Step 1 — Clone BATDiff and your project

Your project is cloned only for the PSNR/SSIM/LPIPS code and the BATDiff patch
(the patch also fixes a Colab packaging import). We still **do not** pass `--xref_image`.

In [ ]:
#@title Step 1 — clone repos and install four packages
import os, subprocess
from pathlib import Path

PROJECT_REPO = "https://github.com/yoyowuyogwrt-hue/3D-OCT-Image-SuperResolution-Benchmark"
BATDIFF_REPO = "https://github.com/MaryamHeidari-1994/BATDiff"

WORK     = Path("/content")
BATDIFF  = WORK / "BATDiff"
PROJECT  = WORK / "project"


def sh(cmd, cwd=None):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(f"command failed with exit code {result.returncode}")


if not BATDIFF.exists():
    sh(f"git clone --depth 1 {BATDIFF_REPO} {BATDIFF}")
if not PROJECT.exists():
    sh(f"git clone --depth 1 {PROJECT_REPO} {PROJECT}")

sh("pip install -q einops ftfy regex PyWavelets lpips")
print("\nBATDiff:", BATDIFF)
print("project:", PROJECT)

In [ ]:
#@title Step 2 — patch BATDiff (compatibility + unused DIP flag)
sh(f"python {PROJECT}/scripts/batdiff_dip_patch.py --batdiff-root {BATDIFF}")
print("Patch applied. This run will omit --xref_image, so x_ref is still bicubic upsample.")

## Step 3 — Upload the photograph pair from your Mac

When the cell runs, click **Choose Files** and select **both**:

- `lr.png`  (64×64, the small image)
- `hr.png`  (2040×1356 original, ground truth; used only for scoring)

Finder path on your Mac: `outputs/sanity/div2k_stride_x2/`

Do **not** upload `dip.png`.

In [ ]:
#@title Step 3 — upload lr.png and hr_crop.png
import shutil
from PIL import Image
from google.colab import files

SR_FACTOR = 4
DATA = WORK / "data" / "natural"
DATA.mkdir(parents=True, exist_ok=True)

print("Upload lr.png and hr.png from outputs/sanity/div2k_filtered_stride_x4/")
uploaded = files.upload()

names = {path.lower(): path for path in uploaded}
if "lr.png" not in names or ("hr.png" not in names and "hr_crop.png" not in names):
    raise FileNotFoundError(
        "Please upload lr.png and hr.png (or hr_crop.png).\n"
        f"You uploaded: {list(uploaded)}"
    )

(DATA / "lr.png").write_bytes(uploaded[names["lr.png"]])
hr_key = names.get("hr.png", names.get("hr_crop.png"))
HR_SRC = WORK / "hr.png"
HR_SRC.write_bytes(uploaded[hr_key])

lr_size = Image.open(DATA / "lr.png").size
hr_size = Image.open(HR_SRC).size
expected_hr = (lr_size[0] * SR_FACTOR, lr_size[1] * SR_FACTOR)
if hr_size != expected_hr:
    raise ValueError(
        f"HR is {hr_size}, but LR {lr_size} at x{SR_FACTOR} expects {expected_hr}.\n"
        "You probably uploaded the old pair. Use outputs/sanity/div2k_stride_x2/."
    )

print(f"LR {lr_size[0]}x{lr_size[1]}")
print(f"HR {hr_size[0]}x{hr_size[1]}  (scoring only; BATDiff never sees this)")

## Step 4 — Configuration

The image is 256×256, so a T4 can usually use a larger `dim` than the OCT 512×1024 run.
Still start with a smoke test.

In [ ]:
#@title Step 4 — configuration
SMOKE_TEST = True  #@param {type:"boolean"}

if SMOKE_TEST:
    DIM, TRAIN_STEPS, TIMESTEPS = 16, 60, 20
else:
    DIM, TRAIN_STEPS, TIMESTEPS = 128, 4000, 100

ATROUS_LEVEL = 6
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

COMMON_FLAGS = (
    f"--mode train "
    f"--image_name lr.png "
    f"--use_atrous --atrous_wavelet b3 "
    f"--atrous_level {ATROUS_LEVEL} "
    f"--sr_factor {SR_FACTOR} "
    f"--dim {DIM} "
    f"--train_num_steps {TRAIN_STEPS} "
    f"--timesteps {TIMESTEPS} "
    f"--save_and_sample_every {max(TRAIN_STEPS, 1)} "
)

print("SMOKE TEST" if SMOKE_TEST else "FULL RUN")
print(f"  dim         {DIM}")
print(f"  train steps {TRAIN_STEPS}")
print(f"  sr_factor   {SR_FACTOR}")
if SMOKE_TEST:
    print("\nSmoke-test pictures may look like noise. That only tests that the code runs.")

In [ ]:
#@title Step 5 — runner
import re, sys, time, subprocess
from pathlib import Path


def find_final_image(scope_dir: Path) -> Path:
    candidates = list((scope_dir / "final_samples").glob("*.png"))
    if not candidates:
        raise FileNotFoundError(f"No samples under {scope_dir / 'final_samples'}")

    def scale_of(path: Path) -> int:
        match = re.search(r"_s(\d+)_", path.name)
        return int(match.group(1)) if match else -1

    finest = max(scale_of(p) for p in candidates)
    at_finest = [p for p in candidates if scale_of(p) == finest]
    return max(at_finest, key=lambda p: p.stat().st_mtime)


def run_batdiff(tag: str, dataset_folder: Path) -> Path:
    scope_dir = RESULTS / tag / tag
    flags = COMMON_FLAGS + f"--scope {tag} "
    flags += f"--dataset_folder {dataset_folder}/ --results_folder {RESULTS / tag} "

    print("=" * 70)
    print(f"RUN {tag}   BATDiff alone (published x_ref = bicubic upsample of LR)")
    print("=" * 70)

    started = time.time()
    process = subprocess.Popen(
        f"python main.py {flags}", shell=True, cwd=BATDIFF, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in process.stdout:
        sys.stdout.write(line)
    process.wait()

    elapsed = (time.time() - started) / 60
    if process.returncode:
        raise RuntimeError(f"run {tag} failed with exit code {process.returncode}")

    output = find_final_image(scope_dir)
    print(f"\nfinished in {elapsed:.1f} min -> {output.name}")
    return output


print("Runner ready.")

In [ ]:
#@title Step 6 — run BATDiff alone (no DIP reference)
out_batdiff = run_batdiff("natural", DATA)

In [ ]:
#@title Step 7 — score against the original photograph
import sys, csv
import numpy as np
from PIL import Image

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.metrics.evaluate import evaluate

hr = np.asarray(Image.open(HR_SRC).convert("RGB"))
target_size = (hr.shape[1], hr.shape[0])
lr = Image.open(DATA / "lr.png").convert("RGB")
bicubic = np.asarray(lr.resize(target_size, Image.BICUBIC))


def load_like_hr(path):
    image = Image.open(path).convert("RGB")
    if image.size != target_size:
        print(f"note: resizing {path.name} from {image.size} to {target_size}")
        image = image.resize(target_size, Image.BICUBIC)
    return np.asarray(image)


methods = {
    "Bicubic upsample": bicubic,
    "BATDiff alone":    load_like_hr(out_batdiff),
}

rows = []
for name, image in methods.items():
    scores = evaluate(hr, image)
    rows.append({"method": name, **scores})

print(f"\n{'method':<20}{'PSNR ↑':>9}{'SSIM ↑':>9}{'LPIPS ↓':>10}")
print("-" * 48)
for row in rows:
    print(f"{row['method']:<20}{row['PSNR']:>9.4f}{row['SSIM']:>9.4f}{row['LPIPS']:>10.4f}")

out_csv = RESULTS / ("scores_smoke.csv" if SMOKE_TEST else "scores.csv")
with out_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["method", "PSNR", "SSIM", "LPIPS"])
    writer.writeheader()
    writer.writerows(rows)
print(f"\nsaved {out_csv}")
if SMOKE_TEST:
    print("SMOKE TEST numbers are meaningless. Set SMOKE_TEST = False and re-run.")

In [ ]:
#@title Step 8 — comparison figure and download
import matplotlib.pyplot as plt

panels = [("HR (ground truth)", hr)] + [
    (row["method"], methods[row["method"]]) for row in rows
]

fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.2))
for axis, (title, image) in zip(axes, panels):
    axis.imshow(image)
    axis.set_title(title, fontsize=11)
    axis.axis("off")
    if title != "HR (ground truth)":
        row = next(r for r in rows if r["method"] == title)
        axis.text(
            0.5, -0.04,
            f"PSNR {row['PSNR']:.2f}  SSIM {row['SSIM']:.3f}  LPIPS {row['LPIPS']:.3f}",
            transform=axis.transAxes, ha="center", va="top", fontsize=8,
        )
fig.tight_layout()

figure_path = RESULTS / ("comparison_smoke.png" if SMOKE_TEST else "comparison.png")
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {figure_path}")

archive = shutil.make_archive(str(WORK / "batdiff_natural_results"), "zip", RESULTS)
print(f"archive: {archive}")
try:
    files.download(archive)
except Exception as error:
    print(f"(automatic download unavailable: {error}; use the Files pane on the left)")

## Troubleshooting

**`No GPU visible`** — Runtime → Change runtime type → T4 GPU.

**`CUDA out of memory`** — in Step 4, change `DIM` from 128 to 64, then
**Runtime → Restart runtime** and run from Step 0 again.

**Upload error / wrong size** — you must upload the stride ×4 pair
(`lr.png` 1020×678, `hr.png` 2040×1356). Do not upload the old 256 crop.

**Smoke test looks like noise** — expected. Only the `SMOKE_TEST = False` run counts.

**Full run still looks like noise** — that is the result to write down. Then BATDiff
is not working on an easy photograph, so do not go back to OCT yet. Paste the figure
and the traceback (if any) into the project chat.

After a successful full run, copy the zip into
`outputs/sanity/div2k_stride_x2/batdiff/` on your Mac.